# 03 · Modelo de Optimización — los 4 enfoques con resultados reales

**Prueba analítica DICAGI 2022 · Bancolombia**

---

## 📌 Objetivo

Resolver el problema con **cuatro enfoques** de complejidad creciente y compararlos con la métrica oficial:

1. 🟢 **Analítico** — sin modelo, solo reglas (lo que un analista de negocio plantearía).
2. 🟡 **Greedy** — heurística por densidad valor/tiempo.
3. 🟠 **MILP exacto** — optimización entera con PuLP/CBC.
4. 🔴 **Simulated Annealing** — física estadística + MCMC (notebook 04 detalle).

**Conclusión adelantada**: los 4 convergen al mismo valor `x/y = 0.4076 = 40.76%` porque el problema **no está saturado por capacidad** — el techo está fijado por la integridad de datos (huérfanos). Esto **no es debilidad del modelo**, es la naturaleza del problema.

## 🧠 Mindset

> *"La elegancia de un modelo no se mide por su sofisticación sino por la simplicidad mínima necesaria para resolver el problema."*

## Pre-requisito

Ejecutar primero el pipeline:
```bash
uv run python -m modelo_capacidad.run_all
```
Esto genera los parquets en `data/processed/` que este notebook consume.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from modelo_capacidad.viz import apply_theme, BANCOLOMBIA_COLORS, BANCOLOMBIA_SEQUENTIAL
apply_theme()

DATA_PROCESSED = REPO_ROOT / 'data' / 'processed'
print('Setup ok.')

---
## 1. Carga de los insumos del modelo (df_ejec, df_ger, tau_cliente)

### 🎯 ¿Qué hacemos?
Leemos los parquets que produjo `tiempo_demanda.py`. Estos son los insumos comunes a los 4 enfoques.

### 👥 Para todo público
Como tener tres planos antes de empezar la obra: el plano de la materia prima (clientes y sus tiempos), el plano de los obreros (ejecutivos), y el plano del jefe (gerentes).

### 🔬 Versión técnica
- `df_ejec`: 370 filas, columnas `cod_ejec_bco, zona, n_clientes, n_a, n_b, n_c, score_medio, t_e, v_e, densidad`.
- `df_ger`: 49 filas, `cod_gte_inv, zona, T_g`.
- `tau_cliente`: 32,394 filas con tiempo demandado por cada cliente.

### ⚛️ Análogo físico
Estamos cargando el **estado del sistema**: partículas (ejecutivos) con masa $t_e$ y carga $v_e$, y los pozos de potencial (gerentes) con profundidad $T_g$.

In [ ]:
df_ejec = pd.read_parquet(DATA_PROCESSED / 'df_ejecutivos.parquet')
df_ger  = pd.read_parquet(DATA_PROCESSED / 'df_gerentes.parquet')
tau_cli = pd.read_parquet(DATA_PROCESSED / 'tau_cliente.parquet')

print(f'df_ejec: {df_ejec.shape}')
print(f'df_ger:  {df_ger.shape}')
print(f'tau:     {tau_cli.shape}')
print()
print(f'Demanda total: {df_ejec["t_e"].sum():>14,.0f} min')
print(f'Oferta total:  {df_ger["T_g"].sum():>14,.0f} min')
print(f'Ratio dem/of:  {df_ejec["t_e"].sum() / df_ger["T_g"].sum():>14.2f}')
df_ejec.head()

### 📏 Lectura de los números

**Demanda 2,592,700 vs oferta 3,720,570 → ratio 0.70**.

Esto significa:
- En conjunto **cabe todo lo asignable** (ratio < 1).
- Pero la utilización promedio será alta (~92%) porque ya filtramos huérfanos.
- El problema NO es "a quién dejar fuera por capacidad", es "a quién recuperar de los huérfanos" (que es proyecto de gobernanza, no de modelo).

---
## 2. Distribución de la densidad valor/tiempo por ejecutivo

### 🎯 ¿Qué hacemos?
Visualizamos $\rho_e = v_e / t_e$ (valor por minuto) — la métrica que el greedy usa para priorizar.

### 🔬 Por qué importa
Si la distribución es muy heterogénea, hay ejecutivos "oro" que conviene asignar primero. Si es homogénea, cualquier orden razonable da el mismo resultado.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Densidad valor/tiempo', 'Tiempo demandado por ejecutivo'))
fig.add_trace(go.Histogram(x=df_ejec['densidad'], nbinsx=40, marker_color=BANCOLOMBIA_COLORS['amarillo']), row=1, col=1)
fig.add_trace(go.Histogram(x=df_ejec['t_e'], nbinsx=40, marker_color=BANCOLOMBIA_COLORS['negro']), row=1, col=2)
fig.update_layout(showlegend=False, title_text='Distribución de ejecutivos asignables (n=370)')
fig.update_xaxes(title_text='valor/tiempo', row=1, col=1)
fig.update_xaxes(title_text='minutos demandados (t_e)', row=1, col=2)
fig.update_yaxes(title_text='# ejecutivos', row=1, col=1)
fig.show()

print('Densidad (v/t):')
print(df_ejec['densidad'].describe(percentiles=[0.5, 0.9, 0.99]).round(2))
print('\nt_e (minutos):')
print(df_ejec['t_e'].describe(percentiles=[0.5, 0.9, 0.99]).round(0))

---
## 3. Comparación de los 4 enfoques

### 🎯 ¿Qué hacemos?
Cargamos `comparativa_enfoques.csv` que generó el pipeline y la visualizamos.

### 🧠 Mindset: ¿qué espero ver?
- MILP debe estar al tope o empatar (es óptimo por construcción).
- Greedy debe acercarse al MILP (1-5% gap típico en GAP).
- SA debe igualar al MILP si está bien parametrizado.
- Analytical (sin modelo) puede quedarse atrás en problemas saturados, pero igualar en holgados.

**Lo que efectivamente vemos**: los 4 al mismo valor 0.4076. Eso indica que el problema no tiene "competencia" por la capacidad — todos los asignables caben.

In [ ]:
comp = pd.read_csv(DATA_PROCESSED / 'comparativa_enfoques.csv')
comp

In [ ]:
fig = go.Figure()
fig.add_bar(
    name='x/y oficial',
    x=comp['enfoque'], y=comp['x/y'],
    text=comp['x/y'].round(4),
    textposition='outside',
    marker_color=BANCOLOMBIA_COLORS['amarillo'],
)
fig.add_hline(y=0.534, line_dash='dot', line_color=BANCOLOMBIA_COLORS['gris_oscuro'],
              annotation_text='Techo absoluto teórico (53.4%)', annotation_position='top left')
fig.add_hline(y=0.408, line_dash='dash', line_color=BANCOLOMBIA_COLORS['negro'],
              annotation_text='Techo real (huérfanos): 40.8%', annotation_position='bottom right')
fig.update_layout(
    title='Métrica oficial x/y por enfoque vs techos teóricos',
    yaxis_title='x/y', xaxis_title=None,
    yaxis_range=[0, 0.6],
)
fig.show()

In [ ]:
# % asignados por categoría
fig = go.Figure()
for cat, color in [('%A', BANCOLOMBIA_COLORS['negro']),
                   ('%B', BANCOLOMBIA_COLORS['amarillo']),
                   ('%C', BANCOLOMBIA_COLORS['gris'])]:
    fig.add_bar(name=cat, x=comp['enfoque'], y=comp[cat], marker_color=color, text=comp[cat])
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(title='% asignado por categoría comercial · por enfoque',
                  yaxis_title='%', barmode='group')
fig.show()

In [ ]:
# Tiempo de cómputo por enfoque (escala log)
fig = px.bar(
    comp, x='enfoque', y='tiempo_s',
    title='Tiempo de cómputo (s) — escala log',
    color='enfoque',
    color_discrete_sequence=[BANCOLOMBIA_COLORS['amarillo'], BANCOLOMBIA_COLORS['amarillo_oscuro'],
                              BANCOLOMBIA_COLORS['negro'], BANCOLOMBIA_COLORS['gris']],
    text='tiempo_s',
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_type='log', yaxis_title='segundos (log)')
fig.show()

### 📏 Conclusión visual

**Los 4 enfoques producen exactamente la misma asignación**. La diferencia está solo en el tiempo de cómputo:
- Analytical y Greedy: ~0.1 s
- MILP: ~0.6 s
- SA: varios segundos por número de pasos MCMC

**Lección**: en problemas no saturados, sofisticación adicional **no aporta** a la métrica. La elección debe basarse en otros criterios (mantenibilidad, diagnóstico, restricciones blandas).

---
## 4. Carga del resultado final y métricas detalladas

In [ ]:
from modelo_capacidad.data.loader import load_clientes
from modelo_capacidad.utils.validators import metrica_oficial

resultado = pd.read_csv(REPO_ROOT / 'resultado_prueba.csv',
                       dtype={'num_doc_cli': 'string', 'cod_ejec_bco': 'string',
                              'num_doc_gte_inv': 'string', 'cod_gte_inv': 'string'})
clientes = load_clientes()

m = metrica_oficial(resultado, clientes)
print(f'Filas en resultado_prueba.csv: {len(resultado):,}')
print(f'Métrica oficial x/y:           {m["metrica_x_y"]:.4f}')
print(f'Clientes asignados:            {m["n_asignados"]:,} ({m["pct_asignados"]:.1f}%)')
print()
print('Por categoría:')
print(f'  A asignados: {m["A_asignados"]:>5,} de {m["A_total"]:>5,} ({m["pct_A"]:.1f}%)')
print(f'  B asignados: {m["B_asignados"]:>5,} de {m["B_total"]:>5,} ({m["pct_B"]:.1f}%)')
print(f'  C asignados: {m["C_asignados"]:>5,} de {m["C_total"]:>5,} ({m["pct_C"]:.1f}%)')

---
## 5. Distribución de carga por gerente — utilización

In [ ]:
# Calcular utilización por gerente: carga asignada vs T_g
carga = (resultado.groupby('cod_gte_inv')['num_doc_cli'].count().reset_index(name='n_clientes'))
carga = carga.merge(
    df_ejec.groupby('cod_ejec_bco').agg(t_e=('t_e', 'first')).reset_index(),
    left_on='cod_gte_inv', right_on='cod_ejec_bco', how='left'
).drop(columns='cod_ejec_bco')

# usado por gerente = sum(t_e de los ejecutivos asignados a ese gerente)
asign_ejec = resultado.drop_duplicates('cod_ejec_bco')[['cod_ejec_bco', 'cod_gte_inv']]
usado = asign_ejec.merge(df_ejec[['cod_ejec_bco', 't_e']], on='cod_ejec_bco')
uso = usado.groupby('cod_gte_inv')['t_e'].sum().reset_index(name='usado')
uso = uso.merge(df_ger[['cod_gte_inv', 'T_g', 'zona']], on='cod_gte_inv')
uso['utilizacion_pct'] = (uso['usado'] / uso['T_g'] * 100).round(1)
uso['holgura'] = uso['T_g'] - uso['usado']
uso = uso.sort_values('utilizacion_pct', ascending=False)

fig = px.bar(
    uso, x='cod_gte_inv', y='utilizacion_pct', color='zona',
    title='Utilización por gerente (asignación final)',
    color_continuous_scale=BANCOLOMBIA_SEQUENTIAL,
)
fig.add_hline(y=100, line_dash='dash', line_color=BANCOLOMBIA_COLORS['rojo_alerta'])
fig.update_layout(xaxis_title='Gerente', yaxis_title='% utilización', xaxis={'showticklabels': False})
fig.show()

print(f'Utilización media: {uso["utilizacion_pct"].mean():.1f}%')
print(f'Utilización min:   {uso["utilizacion_pct"].min():.1f}%')
print(f'Utilización max:   {uso["utilizacion_pct"].max():.1f}%')
print(f'# gerentes saturados (>95%): {(uso["utilizacion_pct"] > 95).sum()}')
print(f'# gerentes con holgura (<70%): {(uso["utilizacion_pct"] < 70).sum()}')

---
## 6. Conclusiones del modelo

1. **Métrica obtenida**: `x/y = 0.4076 (40.76%)` — coincide con el techo teórico predicho en el EDA.
2. **Los 4 enfoques convergen** porque el problema no compite por capacidad — está acotado por integridad de datos.
3. **El modelo más simple basta para producción**: la regla determinista corre en 0.06 s y garantiza factibilidad.
4. **MILP y SA son seguros conceptuales**: validan que la simple solución analítica no deja métrica sobre la mesa.
5. **Próximo paso**: reconciliar los 633 huérfanos en `ecas` para subir el techo de 40.8% → 53.4%.

Ver `notebooks/04_modelo_fisica_estadistica.ipynb` para los diagnósticos físicos del SA (calor específico, susceptibilidad).